# Module 05 — Exercise: Validation and Accuracy Assessment

**DIGIHAZ PhD Course** | Disaster Risk Reduction


## Student Information

- **Name:** Yue Qimin
- **Student ID:** Not provided
- **Date:** 2026-10-01


## 1. Why validation is required

An automated damage map is only as reliable as its verification. Independent reference information can include high-resolution imagery, UAV observations, official assessments or field surveys. For SAR, validation is especially important because radar change can also result from moisture, vegetation, construction, shadow or layover.


## 2. Confusion-matrix metrics

For a binary damage map:

- **Overall Accuracy** = (TP + TN) / N  
- **Precision / User's Accuracy for damaged class** = TP / (TP + FP)  
- **Recall / Producer's Accuracy for damaged class** = TP / (TP + FN)  
- **F1-score** = 2 × Precision × Recall / (Precision + Recall)

Producer's accuracy emphasizes omission error; user's accuracy emphasizes commission error.


In [ ]:
import numpy as np
import pandas as pd

# Example independent validation labels
reference = np.array([1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,1,0])
prediction = np.array([1,1,1,0,1,1,0,1,1,0,0,0,0,1,0,0,0,0,1,0])

tp = int(np.sum((reference==1) & (prediction==1)))
tn = int(np.sum((reference==0) & (prediction==0)))
fp = int(np.sum((reference==0) & (prediction==1)))
fn = int(np.sum((reference==1) & (prediction==0)))

oa = (tp+tn)/len(reference)
precision = tp/(tp+fp)
recall = tp/(tp+fn)
f1 = 2*precision*recall/(precision+recall)

cm = pd.DataFrame([[tp, fn],[fp, tn]],
                  index=["Reference damage","Reference no damage"],
                  columns=["Predicted damage","Predicted no damage"])
display(cm)

pd.Series({
    "Overall accuracy": round(oa,3),
    "User's accuracy / Precision (damage)": round(precision,3),
    "Producer's accuracy / Recall (damage)": round(recall,3),
    "F1-score": round(f1,3)
})


## 3. GEE validation workflow

The lecture recommends:
1. generate stratified random points across the classified map;
2. extract both classified and reference values; and
3. create an error matrix in Google Earth Engine and report the accuracy statistics.

Illustrative GEE logic:

```javascript
var validated = validationPoints.classify(classifier);
var errorMatrix = validated.errorMatrix('reference', 'classification');

print('Confusion matrix', errorMatrix);
print('Overall accuracy', errorMatrix.accuracy());
print('Producer accuracy', errorMatrix.producersAccuracy());
print('User accuracy', errorMatrix.consumersAccuracy());
```


## 4. Sampling and interpretation

Validation samples should be independent of training data and should represent both damaged and undamaged conditions. Stratification is useful when the damaged class is rare. Accuracy values should be reported together with the confusion matrix and with an explanation of reference-data quality.

A high overall accuracy can be misleading in a highly imbalanced dataset, so class-specific precision/recall and F1 are necessary.


## 5. Operational uncertainty

Errors should be investigated spatially. Typical false positives include water, vegetation change, construction activity, layover and shadow. Uncertain or no-data areas should not be forced into a confident damage class.

### Conclusion

Validation converts a visually plausible damage map into an evidence-based product. Independent reference data, transparent metrics and explicit uncertainty are essential for emergency-response use.
